### 제조 센서 데이터의 Datatime과 Timedelta 처리 

- 오전 8시 이후 데이터 찾기
- 월요일 데이터만 찾기 
- 공정 시작과 종료의 차이 계산 
- 10분단위 평균 계산 
- 이전 데이터와의 수집 간격 확인 


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt # 그래프 그릴 때 쓰는 라이브러리 

In [2]:
# pd.to_datetime() 기본과 옵션

time_text = "2026-07-14 08:30:15" # 일반적인 텍스트 

converted_time = pd.to_datetime(
    time_text,
    format = "%Y-%m-%d %H:%M:%S"
)

converted_time

Timestamp('2026-07-14 08:30:15')

In [3]:
type(converted_time)

pandas.Timestamp

In [5]:
# time 텍스트를 to_datetime으로 변환해 보기

time_values = pd.Series([
    "2020-07-14 08:00:00",
    "2020-07-14 08:02:00",
    "2020-13-14 08:04:00",
    "",
    "not-a-time"
])

converted_time = pd.to_datetime(
    time_values,
    format = "%Y-%m-%d %H:%M:%S",
    errors = "coerce"  # 에러가 생기면 -> NaT로 채우는 옵션 
)

option_result_df = pd.DataFrame({
    "original_text" : time_values, # 컬럼명 : 데이터 값
    "converted_time" : converted_time,
    "is_invalid" : converted_time.isna() # NaT로 채워졌는가? 확인하는 것 
})

option_result_df

,original_text,converted_time,is_invalid
0,2020-07-14 08:00:00,2020-07-14 08:00:00,False
1,2020-07-14 08:02:00,2020-07-14 08:02:00,False
2,2020-13-14 08:04:00,NaT,True
3,,NaT,True
4,not-a-time,NaT,True


In [7]:
# 같은 문자열도 format에 따라 다르게 표현 가능

sample_date = pd.Series(["01/02/2026", "05/06/2026"])

month_first = pd.to_datetime(
    sample_date,
    format = "%m/%d/%Y"
)

day_first = pd.to_datetime(
    sample_date,
    format = "%d/%m/%Y"
)

df_sample_date = pd.DataFrame({
    "original" : sample_date,
    "month_first_result" : month_first,
    "day_first_result" : day_first,
})

df_sample_date

,original,month_first_result,day_first_result
0,01/02/2026,2026-01-02,2026-02-01
1,05/06/2026,2026-05-06,2026-06-05


In [8]:
# UTC 초 단위 숫자를 datatime으로 변환 

utc_time = pd.to_datetime("2026-07-14 00:00:00", utc=True)

time_values = pd.Series([0,60,3600]) # 초 분 시간 

time_result = pd.to_datetime(
    time_values, 
    unit = "s", # 초 단위 
    origin = "unix", # UTC 중에 unix라는 표현을 사용 
    utc = True
)

pd.DataFrame({
    "seconds" : time_values,
    "converted_utc" : time_result,
})

,seconds,converted_utc
0,0,1970-01-01 00:00:00+00:00
1,60,1970-01-01 00:01:00+00:00
2,3600,1970-01-01 01:00:00+00:00


In [9]:
# 시계열 데이터 읽어 와서 처리하기

raw_path = r"C:\Users\user\Desktop\python_campus\dev\data\raw\manufacturing_datetime_raw.csv"

df = pd.read_csv(raw_path)

df.head()

,record_id,timestamp_text,machine_id,temperature,pressure,speed,humidity,vibration,current,process_start_text,process_end_text
0,1,2026-07-14 08:00:00,M01,67.84,5.46,1584.7,32.24,1.61,13.65,2026-07-14 08:00:05,2026-07-14 08:00:54
1,2,2026-07-14 08:02:00,M01,70.74,5.19,1423.2,46.40,2.55,13.58,2026-07-14 08:02:05,2026-07-14 08:03:28
2,3,2026-07-14 08:04:00,M01,73.87,4.90,1533.2,37.21,2.60,13.44,2026-07-14 08:04:05,2026-07-14 08:05:10
3,4,2026-07-14 08:06:00,M01,71.26,4.96,1610.0,41.23,2.01,13.08,2026-07-14 08:06:05,2026-07-14 08:07:31
4,5,2026-07-14 08:08:00,M01,73.46,5.34,1538.8,52.71,2.02,12.89,2026-07-14 08:08:05,2026-07-14 08:09:08


In [12]:
df.columns

Index(['record_id', 'timestamp_text', 'machine_id', 'temperature', 'pressure',
       'speed', 'humidity', 'vibration', 'current', 'process_start_text',
       'process_end_text'],
      dtype='str')

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   record_id           180 non-null    int64  
 1   timestamp_text      179 non-null    str    
 2   machine_id          180 non-null    str    
 3   temperature         178 non-null    float64
 4   pressure            178 non-null    float64
 5   speed               180 non-null    float64
 6   humidity            180 non-null    float64
 7   vibration           180 non-null    float64
 8   current             180 non-null    float64
 9   process_start_text  180 non-null    str    
 10  process_end_text    180 non-null    str    
dtypes: float64(6), int64(1), str(4)
memory usage: 15.6 KB


In [14]:
# 시간 변환과 .dt 기능 확인 

clean_df = df.copy()

clean_df["timestamp"] = pd.to_datetime(
    clean_df["timestamp_text"],
    format = "%Y-%m-%d %H:%M:%S",
    errors ="coerce"
)

clean_df["process_start_text"] = pd.to_datetime(
    clean_df["process_start_text"],
    format = "%Y-%m-%d %H:%M:%S",
    errors ="coerce"
)

clean_df["process_end_text"] = pd.to_datetime(
    clean_df["process_end_text"],
    format = "%Y-%m-%d %H:%M:%S",
    errors ="coerce"
)

clean_df.head(10)

,record_id,timestamp_text,machine_id,temperature,pressure,speed,humidity,vibration,current,process_start_text,process_end_text,timestamp
0,1,2026-07-14 08:00:00,M01,67.84,5.46,1584.7,32.24,1.61,13.65,2026-07-14 08:00:05,2026-07-14 08:00:54,2026-07-14 08:00:00
1,2,2026-07-14 08:02:00,M01,70.74,5.19,1423.2,46.40,2.55,13.58,2026-07-14 08:02:05,2026-07-14 08:03:28,2026-07-14 08:02:00
2,3,2026-07-14 08:04:00,M01,73.87,4.90,1533.2,37.21,2.60,13.44,2026-07-14 08:04:05,2026-07-14 08:05:10,2026-07-14 08:04:00
3,4,2026-07-14 08:06:00,M01,71.26,4.96,1610.0,41.23,2.01,13.08,2026-07-14 08:06:05,2026-07-14 08:07:31,2026-07-14 08:06:00
4,5,2026-07-14 08:08:00,M01,73.46,5.34,1538.8,52.71,2.02,12.89,2026-07-14 08:08:05,2026-07-14 08:09:08,2026-07-14 08:08:00
5,6,2026-07-14 08:10:00,M01,68.74,5.42,1601.6,41.43,1.82,12.51,2026-07-14 08:10:05,2026-07-14 08:11:13,2026-07-14 08:10:00
6,7,2026-07-14 08:12:00,M01,74.97,5.39,1440.1,43.16,2.25,13.76,2026-07-14 08:12:05,2026-07-14 08:13:18,2026-07-14 08:12:00
7,8,2026-07-14 08:14:00,M01,75.49,5.28,1561.1,42.34,2.33,14.26,2026-07-14 08:14:05,2026-07-14 08:15:23,2026-07-14 08:14:00
8,9,2026-07-14 08:16:00,M01,70.72,5.04,1442.5,40.62,2.87,12.46,2026-07-14 08:16:05,2026-07-14 08:17:13,2026-07-14 08:16:00
9,10,2026-07-14 08:18:00,M01,75.87,4.61,1469.9,42.81,2.46,14.35,2026-07-14 08:18:05,2026-07-14 08:19:25,2026-07-14 08:18:00


In [15]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   record_id           180 non-null    int64         
 1   timestamp_text      179 non-null    str           
 2   machine_id          180 non-null    str           
 3   temperature         178 non-null    float64       
 4   pressure            178 non-null    float64       
 5   speed               180 non-null    float64       
 6   humidity            180 non-null    float64       
 7   vibration           180 non-null    float64       
 8   current             180 non-null    float64       
 9   process_start_text  180 non-null    datetime64[us]
 10  process_end_text    180 non-null    datetime64[us]
 11  timestamp           177 non-null    datetime64[us]
dtypes: datetime64[us](3), float64(6), int64(1), str(2)
memory usage: 17.0 KB


In [16]:
# 시간 변환에 실패한 행을 확인하기 

invalid_time_df = clean_df[
    clean_df["timestamp"].isna() # NaT 확인 
    ].copy()

invalid_time_df[["record_id", "timestamp_text", "machine_id"]]

,record_id,timestamp_text,machine_id
10,11,2026-13-40 08:20:00,M01
80,81,NaN,M02
150,151,not-a-time,M03


In [18]:
# 타임 데이터에서 연, 월, 일, 시, 분, 요일을 추출하기

clean_df["year"] = clean_df["timestamp"].dt.year        # 연 추출
clean_df["month"] = clean_df["timestamp"].dt.month      # 월 추출
clean_df["day"] = clean_df["timestamp"].dt.day          # 일 추출
clean_df["hour"] = clean_df["timestamp"].dt.hour        # 시 추출
clean_df["minute"] = clean_df["timestamp"].dt.minute    # 분 추출
clean_df["weekday"] = clean_df["timestamp"].dt.weekday  # 요일 추출
clean_df["date"] = clean_df["timestamp"].dt.date        # 날짜 추출 (년-월-일)

weekday_map = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}
clean_df["weekday_ko"] = clean_df["weekday"].map(weekday_map) # 매핑

clean_df[["timestamp", "year", "month", "day", "hour", "minute", "weekday", "date", "weekday_ko"]].head()

,timestamp,year,month,day,hour,minute,weekday,date,weekday_ko
0,2026-07-14 08:00:00,2026.0,7.0,14.0,8.0,0.0,1.0,2026-07-14,화
1,2026-07-14 08:02:00,2026.0,7.0,14.0,8.0,2.0,1.0,2026-07-14,화
2,2026-07-14 08:04:00,2026.0,7.0,14.0,8.0,4.0,1.0,2026-07-14,화
3,2026-07-14 08:06:00,2026.0,7.0,14.0,8.0,6.0,1.0,2026-07-14,화
4,2026-07-14 08:08:00,2026.0,7.0,14.0,8.0,8.0,1.0,2026-07-14,화


In [19]:
# 날짜, 시간 조건으로 데이터 찾기 

# 오전 8시 30분 부터 9시까지의 데이터만 찾기 

start_time = pd.Timestamp("2026-07-14 08:30:00")
end_time = pd.Timestamp("2026-07-14 09:00:00")

time_filtered_df = clean_df[
    clean_df["timestamp"].between(start_time, end_time)
].copy()

time_filtered_df.head(10)

,record_id,timestamp_text,machine_id,temperature,pressure,speed,humidity,vibration,current,process_start_text,process_end_text,timestamp,year,month,day,hour,minute,weekday,date,weekday_ko
15,16,2026-07-14 08:30:00,M01,73.60,4.88,1466.0,48.50,2.04,14.39,2026-07-14 08:30:05,2026-07-14 08:31:09,2026-07-14 08:30:00,2026.0,7.0,14.0,8.0,30.0,1.0,2026-07-14,화
16,17,2026-07-14 08:32:00,M01,71.18,4.87,1469.5,46.20,1.42,14.02,2026-07-14 08:32:05,2026-07-14 08:33:29,2026-07-14 08:32:00,2026.0,7.0,14.0,8.0,32.0,1.0,2026-07-14,화
17,18,2026-07-14 08:34:00,M01,72.95,4.99,1369.9,42.36,1.96,13.78,2026-07-14 08:34:05,2026-07-14 08:35:12,2026-07-14 08:34:00,2026.0,7.0,14.0,8.0,34.0,1.0,2026-07-14,화
18,19,2026-07-14 08:36:00,M01,78.41,5.12,1407.9,42.90,2.30,15.13,2026-07-14 08:36:05,2026-07-14 08:37:05,2026-07-14 08:36:00,2026.0,7.0,14.0,8.0,36.0,1.0,2026-07-14,화
19,20,2026-07-14 08:38:00,M01,75.34,5.32,1631.7,36.06,1.91,12.39,2026-07-14 08:38:05,2026-07-14 08:39:21,2026-07-14 08:38:00,2026.0,7.0,14.0,8.0,38.0,1.0,2026-07-14,화
20,21,2026-07-14 08:40:00,M01,66.49,5.42,1480.0,34.65,1.74,13.88,2026-07-14 08:40:05,2026-07-14 08:41:39,2026-07-14 08:40:00,2026.0,7.0,14.0,8.0,40.0,1.0,2026-07-14,화
21,22,2026-07-14 08:42:00,M01,75.35,5.90,1762.2,44.07,1.75,10.94,2026-07-14 08:42:05,2026-07-14 08:43:19,2026-07-14 08:42:00,2026.0,7.0,14.0,8.0,42.0,1.0,2026-07-14,화
22,23,2026-07-14 08:44:00,M01,68.75,5.05,1444.9,41.30,2.68,13.69,2026-07-14 08:44:05,2026-07-14 08:45:30,2026-07-14 08:44:00,2026.0,7.0,14.0,8.0,44.0,1.0,2026-07-14,화
23,24,2026-07-14 08:46:00,M01,71.37,4.84,1349.3,39.57,2.18,15.62,2026-07-14 08:46:05,2026-07-14 08:47:03,2026-07-14 08:46:00,2026.0,7.0,14.0,8.0,46.0,1.0,2026-07-14,화
24,25,2026-07-14 08:48:00,M01,75.93,5.03,1393.4,37.17,1.87,16.05,2026-07-14 08:48:05,2026-07-14 08:49:36,2026-07-14 08:48:00,2026.0,7.0,14.0,8.0,48.0,1.0,2026-07-14,화


In [20]:
time_filtered_df.tail()

,record_id,timestamp_text,machine_id,temperature,pressure,speed,humidity,vibration,current,process_start_text,process_end_text,timestamp,year,month,day,hour,minute,weekday,date,weekday_ko
145,146,2026-07-14 08:50:00,M03,75.55,5.37,1608.0,43.45,2.04,13.90,2026-07-14 08:50:05,2026-07-14 08:50:58,2026-07-14 08:50:00,2026.0,7.0,14.0,8.0,50.0,1.0,2026-07-14,화
146,147,2026-07-14 08:52:00,M03,73.53,3.92,1344.9,44.26,2.41,12.11,2026-07-14 08:52:05,2026-07-14 08:53:31,2026-07-14 08:52:00,2026.0,7.0,14.0,8.0,52.0,1.0,2026-07-14,화
147,148,2026-07-14 08:54:00,M03,69.15,5.68,1456.4,53.21,2.20,13.99,2026-07-14 08:54:05,2026-07-14 08:55:39,2026-07-14 08:54:00,2026.0,7.0,14.0,8.0,54.0,1.0,2026-07-14,화
148,149,2026-07-14 08:56:00,M03,72.52,4.85,1490.1,41.82,1.59,13.19,2026-07-14 08:56:05,2026-07-14 08:57:31,2026-07-14 08:56:00,2026.0,7.0,14.0,8.0,56.0,1.0,2026-07-14,화
149,150,2026-07-14 08:58:00,M03,69.03,5.52,1503.1,40.59,2.15,13.77,2026-07-14 08:58:05,2026-07-14 08:59:00,2026-07-14 08:58:00,2026.0,7.0,14.0,8.0,58.0,1.0,2026-07-14,화


In [23]:
# 특정 장비와 시간 조건을 함께 사용하여 필터링 하기

machin_time_df = clean_df[
    (clean_df["machine_id"] == "M02")
    & (clean_df["hour"] == 9)
    ].copy()

machin_time_df[["timestamp", "machine_id", "temperature", "pressure"]].head()

,timestamp,machine_id,temperature,pressure
90,2026-07-14 09:00:00,M02,74.86,5.90
91,2026-07-14 09:02:00,M02,67.11,5.29
92,2026-07-14 09:04:00,M02,75.25,5.28
93,2026-07-14 09:06:00,M02,69.49,4.98
94,2026-07-14 09:08:00,M02,67.86,5.28


In [24]:
machin_time_df[["timestamp", "machine_id", "temperature", "pressure"]].describe()

,timestamp,temperature,pressure
count,30,30.000000,29.000000
mean,2026-07-14 09:29:00,72.346333,5.266207
min,2026-07-14 09:00:00,62.350000,4.300000
25%,2026-07-14 09:14:30,69.972500,4.980000
50%,2026-07-14 09:29:00,71.975000,5.280000
75%,2026-07-14 09:43:30,74.770000,5.620000
max,2026-07-14 09:58:00,80.210000,6.030000
std,NaN,3.886416,0.403143


timestamp에서 second를 추출해 새 열을 추가하세요.

In [2]:
clean_df["second"] = clean_df["timestamp"].dt.second

clean_df

NameError: name 'clean_df' is not defined